# IO handler from time series

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import glob
import os
import shutil

import affine
import numpy as np
import pandas as pd
import rioxarray as rio  # noqa: F401
import xarray as xr
from pyproj import CRS

In [ ]:
import evaspa.timeseries.io_handler as ioh
import evaspa.timeseries.status_handler as sh
from evaspa.timeseries.constant import TimeSeriesVar as TSVar

### Create time series

In [ ]:
# Setup data
window_size = 7  # Time
x_size = 1
y_size = 2
today = dt.datetime.now(tz=dt.timezone.utc).date()
dates = np.array(pd.date_range(end=today, periods=window_size).to_list())
# Define transform: (origin_x, origin_y), pixel size = 0.1 degree
transform = affine.Affine.translation(0, 40) * affine.Affine.scale(0.1, -0.1)
# Generate x/y coordinates from transform
x_coords = [(transform * (i, 0))[0] for i in range(x_size)]  # x: lon
y_coords = [(transform * (0, j))[1] for j in range(y_size)]  # y: lat
# Create a dataset
et_ts = xr.Dataset(
    {
        TSVar.ET.value: (
            ["time", "y", "x"],
            np.transpose(
                np.array(
                    [
                        [
                            [1.2, 1.4, 1.6, 1.8, 1.8, 1.8],
                            [1.4, 1.4, 1.4, 1.4, 1.4, 1.4],
                        ]
                    ]
                ),
                (2, 1, 0),
            ),
        ),
        TSVar.FLAGS.value: (
            ["time", "y", "x"],
            np.transpose(
                np.array(
                    [
                        [
                            [0, 386, 386, 0, 131, 275],
                            [132, 0, 131, 259, 387, 531],
                        ]
                    ],
                    dtype=sh.STATUS_TYPE,
                ),
                (2, 1, 0),
            ),
        ),
    },
    coords={
        "time": dates[:-1],
        "x": x_coords,
        "y": y_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
et_ts = et_ts.rio.write_crs(CRS(4236))
et_ts = et_ts.rio.write_transform(transform)
# Create a rdaiation dataset
radiation_ts = xr.Dataset(
    {
        TSVar.RADIATION.value: (
            ["time", "y", "x"],
            np.transpose(
                np.array(
                    [
                        [
                            [200, 200, 200, 200, 200, 200, 200],
                            [200, 200, 200, 200, 200, 200, 200],
                        ]
                    ]
                ),
                (2, 1, 0),
            ),
        ),
    },
    coords={
        "time": dates,
        "x": x_coords,
        "y": y_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
radiation_ts = radiation_ts.rio.write_crs(CRS(4236))
radiation_ts = radiation_ts.rio.write_transform(transform)
# Create a dataset for ET signe dates
acquisition_dates = [1, 3, 5, 6]
feed_dates = dates[acquisition_dates]
feed = xr.Dataset(
    {
        TSVar.ET.value: (
            ["time", "y", "x"],
            np.transpose(
                np.array(
                    [[[np.nan, 1.8, np.nan, 2.4], [1.4, np.nan, 2.2, np.nan]]]
                ),
                (2, 1, 0),
            ),
        ),
        TSVar.FLAGS.value: (
            ["time", "y", "x"],
            np.transpose(np.array([[[0, 1, 0, 1], [1, 0, 1, 0]]]), (2, 1, 0)),
        ),
    },
    coords={
        "time": feed_dates,
        "x": x_coords,
        "y": y_coords,
    },
)
# Add CRS and transform metadata (compatible with rioxarray)
feed = feed.rio.write_crs(CRS(4236))
feed = feed.rio.write_transform(transform)

### Write time series 

In [ ]:
output_dir = os.path.join(os.getcwd(), "out")
if os.path.isdir(output_dir):
    shutil.rmtree(output_dir)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
ioh.write_timeseries(
    et_ts,
    root_name="et_time_series",
    directory=output_dir,
)
ioh.write_timeseries(radiation_ts, root_name="radiation", directory=output_dir)
ioh.write_timeseries(feed, root_name="et_single_date", directory=output_dir)

### Read time series

In [ ]:
# Generate lists of file paths
et_ts_files = [
    os.path.join(output_dir, file)
    for file in glob.glob("et_time_series_*.tif", root_dir=output_dir)
]
radiation_files = [
    os.path.join(output_dir, file)
    for file in glob.glob("radiation_*.tif", root_dir=output_dir)
]
et_files = [
    os.path.join(output_dir, file)
    for file in glob.glob("et_single_date_*.tif", root_dir=output_dir)
]

In [ ]:
print("ET time series = ", et_ts_files)
print("Radiation = ", radiation_files)
print("ET single date = ", et_files)

In [ ]:
et_ts.sizes

In [ ]:
dates

In [ ]:
et_ts.reindex({TSVar.TIME.value: dates}, method=None)

In [ ]:
et_ts = ioh.read_et_time_series(et_ts_files, dates=dates)

In [ ]:
et_ts

In [ ]:
radiation_ts = ioh.read_radiation_time_series(radiation_files)

In [ ]:
radiation_ts

In [ ]:
et_feed = ioh.read_et_single_date(et_files)

In [ ]:
et_feed

### Read input data

In [ ]:
config = {
    "dates": [
        "2025-08-23",
        "2025-08-24",
        "2025-08-25",
        "2025-08-26",
        "2025-08-27 00:00:00",
        "2025-08-28",
        "2025-08-29 00:00:00",
    ],
    "et_time_series": et_ts_files,
    "radiation": radiation_files,
    "et_single_date": et_files,
}

In [ ]:
ioh.TimeSeriesInputConfig.model_validate(config)

In [ ]:
et_ts, radiation_ts, et_sd = ioh.read_input(config)

In [ ]:
print(et_ts)
print(radiation_ts)
print(et_sd)

In [ ]:
config = {
    "et_time_series": et_ts_files,
}

In [ ]:
ioh.TimeSeriesInputConfig.model_validate(config)

In [ ]:
et_ts, radiation_ts, et_sd = ioh.read_input(config)

In [ ]:
print(et_ts)
print(radiation_ts)
print(et_sd)